## Trabajo Práctico Grupal · Cámara de Diputados, 2022–2026


In [ ]:
import collections
import math
import random
import re
import unicodedata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize
from scipy.special import log_ndtr
import igraph as ig
from wordcloud import WordCloud

## Colores por bloque

Cada bloque tiene un color fijo en todo el trabajo, elegido según el color con
el que se lo identifica en la política argentina: violeta para La Libertad
Avanza, amarillo para el PRO, celeste para el peronismo (Frente de Todos y su
sucesor, Unión por la Patria), rojo para la UCR, bordó para el Frente de
Izquierda y naranja para Provincias Unidas. Los bloques sin un color
establecido (desprendimientos de la UCR, bloques provinciales) van en tonos que
no compitan con esos: rosados los que salen del radicalismo, neutros el resto.

El celeste del peronismo y el rojo de la UCR son más claro y más oscuro,
respectivamente, que el azul y rojo de la escala izquierda/derecha del mapa de
proyectos (`IZQUIERDA` en rojo, `DERECHA` en azul), para que no se confundan
con esa polaridad.

In [ ]:
NOMBRE_BLOQUE = {
    "Frente De Todos": "Frente de Todos",
    "Union Por La Patria": "Unión por la Patria",
    "La Libertad Avanza": "La Libertad Avanza",
    "Pro": "PRO",
    "Ucr": "UCR",
    "Ucr - Union Civica Radical": "UCR",
    "Evolucion Radical": "Evolución Radical",
    "Coalicion Civica": "Coalición Cívica",
    "Encuentro Federal": "Encuentro Federal",
    "Democracia Para Siempre": "Democracia para Siempre",
    "Provincias Unidas": "Provincias Unidas",
    "Innovacion Federal": "Innovación Federal",
}

COLOR_BLOQUE = {
    "La Libertad Avanza": "#7b3fb5",
    "PRO": "#f0b800",
    "Frente de Todos": "#5bb8ec",
    "Unión por la Patria": "#5bb8ec",
    "UCR": "#c8102e",
    "Frente de Izquierda": "#7a1f2b",
    "Provincias Unidas": "#f07c1e",
    "Coalición Cívica": "#1baf7a",
    "Evolución Radical": "#e87ba4",
    "Democracia para Siempre": "#c45a8c",
    "Encuentro Federal": "#6f8296",
    "Innovación Federal": "#a3845c",
    "Otros bloques": "#c3c2b7",
}

## Cargar los puntos ideales


In [ ]:
df = pd.read_csv("datos/idps.csv")
df.head()

## Orientar el PCA

El signo de `idp_pca` es arbitrario por año. Lo reorientamos acá
contra `idp_bayes`, que sí viene fijado contra legisladores de referencia de
izquierda/derecha: si la correlación
entre los dos métodos da negativa para un año, se invierte el signo de
`idp_pca` ese año.

In [ ]:
def orientar_pca(df):
    df = df.copy()
    for anio, g in df.groupby("anio"):
        r = g["idp_pca"].corr(g["idp_bayes"])
        if r is not None and r < 0:
            df.loc[g.index, "idp_pca"] = -df.loc[g.index, "idp_pca"]
    return df


df = orientar_pca(df)

## Bloque de cada diputado

El bloque de cada legislador ese año sale de `legisladores_bloque_provincia.csv`.
Todos los bloques del Frente de Izquierda (PTS, Partido Obrero, Izquierda
Socialista, MST: 1 o 2 bancas cada uno, con nombres que cambian de un año a
otro) se agrupan bajo "Frente de Izquierda" —así se los suele nombrar en
conjunto— para que el color tenga sentido como categoría. El resto de los
bloques con menos de 10 legisladores *ese año* va a "Otros bloques", igual que
los pocos legisladores que no aparecen en ese archivo. Los colores son los de
`COLOR_BLOQUE` (definidos al principio), así que cada bloque se ve igual en
todas las solapas y en todos los gráficos.

In [ ]:
bloques = pd.read_csv("datos/legisladores_bloque_provincia.csv")

df = df.merge(bloques[["legislador", "yr", "bloque"]], left_on=["nombre", "anio"], right_on=["legislador", "yr"],
              how="left").drop(columns=["legislador", "yr"])
df["bloque"] = df["bloque"].where(~df["bloque"].str.contains("izquierda", case=False, na=False), "Frente de Izquierda")
df["bloque"] = df["bloque"].replace(NOMBRE_BLOQUE)
grandes = df.groupby(["anio", "bloque"])["nombre"].transform("size") >= 10
df["bloque_mostrado"] = df["bloque"].where(grandes | (df["bloque"] == "Frente de Izquierda"), "Otros bloques")
assert df["bloque_mostrado"].isin(COLOR_BLOQUE).all(), set(df["bloque_mostrado"]) - set(COLOR_BLOQUE)
df.groupby("anio")["bloque_mostrado"].value_counts().unstack(0).fillna(0).astype(int)

## Puntos ideales por año: Bayes y PCA

In [ ]:
IZQUIERDA, DERECHA = "#e34948", "#2a78d6"
WHISKER, BASELINE, GRID = "#898781", "#c3c2b7", "#e1e0d9"
SURFACE, PAGE = "#fcfcfb", "#f9f9f7"
TEXT, TEXT2, MUTED = "#0b0b0b", "#52514e", "#898781"
FONT = "system-ui, -apple-system, \'Segoe UI\', sans-serif"


def panel(fig, row, col, g, columna, sd=None):
    """Un panel: barras divergentes desde 0, ordenadas por valor y coloreadas
    por bloque, con opcionalmente un whisker +-sd (Bayes) y hover de nombre,
    bloque e idp."""
    g = g.sort_values(columna)
    bloque = g["bloque"].fillna("sin bloque")
    kw = dict(
        x=g[columna], y=list(range(len(g))), orientation="h",
        marker=dict(color=g["bloque_mostrado"].map(COLOR_BLOQUE), line=dict(width=0)),
        showlegend=False,
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
    )
    if sd is not None:
        kw["error_x"] = dict(type="data", array=g[sd], color=WHISKER, thickness=1, width=0)
        kw["customdata"] = list(zip(g["nombre"], bloque, g[sd]))
        kw["hovertemplate"] = "<b>%{customdata[0]}</b><br>%{customdata[1]}<br>idp = %{x:.2f} (\u00b1%{customdata[2]:.2f})<extra></extra>"
    else:
        kw["customdata"] = list(zip(g["nombre"], bloque))
        kw["hovertemplate"] = "<b>%{customdata[0]}</b><br>%{customdata[1]}<br>idp = %{x:.2f}<extra></extra>"
    fig.add_trace(go.Bar(**kw), row=row, col=col)

In [ ]:
anios = sorted(df["anio"].unique())

bayes_domain = math.ceil((df["idp_bayes"].abs() + df["sd_bayes"]).max() * 10) / 10
pca_domain = 1.08

fig = make_subplots(
    rows=1, cols=2,
    column_titles=["Bayes · media posterior", "PCA · reescalado a [-1, 1], orientado contra Bayes"],
    horizontal_spacing=0.05,
)

trazas_fig = {}
for anio in anios:
    g = df[df["anio"] == anio]
    trazas_fig[anio] = [len(fig.data), len(fig.data) + 1]
    panel(fig, 1, 1, g, "idp_bayes", sd="sd_bayes")
    panel(fig, 1, 2, g.dropna(subset=["idp_pca"]), "idp_pca")
    # leyenda del año: trazas invisibles, una por bloque
    for bloque, gb in g.groupby("bloque_mostrado"):
        trazas_fig[anio].append(len(fig.data))
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                                  marker=dict(size=10, color=COLOR_BLOQUE[bloque]),
                                  name=f"{bloque} ({len(gb)})",
                                  legendrank=1000 if bloque == "Otros bloques" else gb["idp_bayes"].median() * 100 + 500),
                      row=1, col=1)

fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                          line=dict(color=WHISKER, width=2),
                          name="\u00b1 1 desvio (incertidumbre, solo Bayes)", legendrank=1001), row=1, col=1)


def visibles(anio):
    return [i in trazas_fig[anio] for i in range(len(fig.data) - 1)] + [True]


inicial = anios[-1]
for traza, visible in zip(fig.data, visibles(inicial)):
    traza.visible = visible

botones = [dict(label=str(a), method="update", args=[{"visible": visibles(a)}]) for a in anios]

fig.update_xaxes(range=[-bayes_domain, bayes_domain], zeroline=True,
                  zerolinecolor=BASELINE, zerolinewidth=1, gridcolor=GRID,
                  tickfont=dict(size=10, color=MUTED), row=1, col=1)
fig.update_xaxes(range=[-pca_domain, pca_domain], zeroline=True,
                  zerolinecolor=BASELINE, zerolinewidth=1, gridcolor=GRID,
                  tickfont=dict(size=10, color=MUTED), row=1, col=2)
fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, autorange="reversed")

fig.update_annotations(font=dict(size=13, color=TEXT2))

fig.update_layout(
    title=dict(text="Puntos ideales \u00b7 C\u00e1mara de Diputados, 2022\u20132026", x=0.01, y=0.98, yanchor="top",
               font=dict(size=20, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=900, width=1000,
    margin=dict(l=30, r=20, t=150, b=30),
    legend=dict(x=1.0, y=1.0, xanchor="left", yanchor="top", font=dict(size=11)),
    bargap=0.1,
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, active=anios.index(inicial),
                      x=0.01, xanchor="left", y=1.08, yanchor="bottom", showactive=True,
                      pad=dict(r=4, t=0), bgcolor=SURFACE, bordercolor=GRID,
                      font=dict(size=11, color=TEXT2))],
)

fig.show(config={"displaylogo": False, "scrollZoom": False})

**Desvíos grandes que no son ideología: Massa (2022) y Marcela Herrera (2023).**
En 2022, Sergio Massa tiene el desvío más grande de la Cámara (`sd_bayes` ≈ 0.80)
con un `idp_bayes` cerca de 0. No es que sea ambiguo: fue presidente de la Cámara
hasta agosto, y el presidente no vota, así que en sus registros solo figura
`PRESIDENTE` o `AUSENTE`, sin un solo Sí o No. Marcela Herrera, en 2023, aparece
en una sola votación (la última del año) y es unánime, y las unánimes quedan
afuera del modelo. En los dos casos el modelo no tiene ningún voto útil, y lo que
muestra es el prior: un punto cerca del centro con una incertidumbre del ancho
de todo el eje. Esa es una ventaja de Bayes sobre PCA: el PCA igual le asigna a
Herrera un valor (−0.50) que parece tan firme como cualquier otro, porque sale de
imputar los votos que le faltan. Bayes, en cambio, avisa que no sabe.

**El Frente de Izquierda en el centro (2022): el eje mide cercanía al gobierno.**
En 2022, Bregman, Del Caño y Del Pla quedan en `idp_bayes` ≈ +0.37, apenas a la
derecha del cero (puestos 137–140 de 259), mientras que Máximo Kirchner está en
−1.11 y Milei en +2.37. No es que el FIT esté "cerca de Milei": Bregman vota igual
que Kirchner en el 74% de las votaciones que comparten y que Milei en el 46%. Pero
también coincide un 74% con Negri (UCR), y eso la deja en el medio. El FIT
acompaña al Frente de Todos en muchas leyes sociales, pero vota en contra del
gobierno en las económicas centrales: Presupuesto 2023, Consenso Fiscal,
prórroga de impuestos, promoción automotriz, biotecnología. En esas votaciones
Bregman y Milei votan No y Kirchner Sí; el FIT se opone por izquierda y Milei por
derecha, pero el voto es el mismo. Un modelo de una sola dimensión no tiene cómo
representar "opositor por izquierda", y un diputado que a veces vota con el
oficialismo y a veces con la oposición termina en el promedio. Es una limitación
conocida (*ends against the middle*: los dos extremos votan juntos contra el
centro, por motivos opuestos), y muestra que en estos años el eje mide sobre todo
la distancia al oficialismo de turno. Por eso el FIT se corre a medida que cambia
el gobierno: en 2023 queda casi en el centro (−0.14), en 2024 y 2025 ya está del
lado izquierdo junto a Unión por la Patria, y en 2026 sus tres diputados son los
tres más a la izquierda de la Cámara. Con Milei en el gobierno, oponerse al
oficialismo y ser de izquierda pasan a ser lo mismo.

## Tabla de posiciones: los 10 desvíos más grandes de cada año

Los diez diputados con `sd_bayes` más alto en cada año, sin los casos como
Massa o Herrera: se dejan afuera los que tienen menos de 10 **votos útiles**
ese año, es decir, votos Sí o No en votaciones que no fueron unánimes (las
únicas que usa el modelo; ver `preparar_pscl()` en `03_puntos_ideales.py`).
El color de cada celda es el de su bloque.

In [ ]:
MIN_VOTOS_UTILES = 10

ses = pd.read_csv("datos/sesiones_dt.csv.gz", dtype={"legislador": str}, keep_default_na=False)

si_no = ses[ses["voto"].isin(["AFIRMATIVO", "NEGATIVO"])]
divididas = si_no.groupby("id_votacion")["voto"].nunique().loc[lambda n: n == 2].index
utiles = si_no[si_no["id_votacion"].isin(divididas)].groupby(["legislador", "yr"]).size()
df["votos_utiles"] = utiles.reindex(pd.MultiIndex.from_frame(df[["nombre", "anio"]])).fillna(0).astype(int).to_numpy()

top_desvio = (df[df["votos_utiles"] >= MIN_VOTOS_UTILES]
              .sort_values("sd_bayes", ascending=False)
              .groupby("anio").head(10))
excluidos = df[df["votos_utiles"] < MIN_VOTOS_UTILES].sort_values(["anio", "votos_utiles"])[
    ["anio", "nombre", "votos_utiles", "idp_bayes", "sd_bayes"]]
print(f"Fuera de la tabla de posiciones (menos de {MIN_VOTOS_UTILES} votos útiles): {len(excluidos)} diputados")
excluidos

In [ ]:
def suave(color, alfa=0.25):
    r, g, b = (int(color[i:i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r}, {g}, {b}, {alfa})"


textos, fondos = [list(range(1, 11))], [[SURFACE] * 10]
for anio, g in top_desvio.groupby("anio"):
    textos.append([f"<b>{n.title()}</b><br>{b} · sd {sd:.2f} · idp {idp:+.2f}"
                   for n, b, sd, idp in zip(g["nombre"], g["bloque"].fillna("sin bloque"), g["sd_bayes"], g["idp_bayes"])])
    fondos.append([suave(COLOR_BLOQUE[b]) for b in g["bloque_mostrado"]])

fig_desvio = go.Figure(go.Table(
    columnwidth=[0.25] + [1] * len(anios),
    header=dict(values=["#"] + [f"<b>{a}</b>" for a in anios], fill_color=PAGE, line_color=GRID,
                font=dict(size=13, color=TEXT), align="left", height=32),
    cells=dict(values=textos, fill_color=fondos, line_color=SURFACE, align="left", height=48,
               font=dict(size=11, color=TEXT)),
))
fig_desvio.update_layout(
    title=dict(text=f"Los 10 desvíos más grandes de cada año · con al menos {MIN_VOTOS_UTILES} votos útiles",
               x=0.01, font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT), paper_bgcolor=PAGE,
    width=1300, height=640, margin=dict(l=20, r=20, t=60, b=20),
)
fig_desvio.show(config={"displaylogo": False})

**Cómo leer estos desvíos.** Casi todos los de la tabla tienen `|idp| > 1`: están
en los **bordes** del eje, no en el medio. Es un efecto conocido del modelo: un
diputado queda ubicado por las votaciones que lo separan de sus vecinos, y los
que están en un extremo no tienen a nadie más allá. Si todo el ala dura del
oficialismo (o de la oposición) vota siempre igual, el modelo sabe que están
"muy a la derecha" (o a la izquierda), pero no puede distinguir si uno está en
1.2 o en 1.6, porque no hay ninguna votación que los parta. El desvío alto no
significa que voten de forma errática; significa lo contrario: votan tan en
bloque que no hay datos para ordenarlos entre sí.

Leído así, la tabla cuenta quién fue el polo más compacto cada año:

- **2022 y 2023:** todo el top 10 es del Frente de Todos, en el extremo
  izquierdo. El ala dura del oficialismo votaba como un solo bloque.
- **2024:** el top se reparte entre Unión por la Patria y La Libertad Avanza,
  los dos polos del primer año de Milei. Los desvíos son más chicos (≈ 0.3–0.4)
  porque fue el año con más votaciones (más de 150 votos útiles por diputado):
  con más datos, todas las estimaciones se afinan.
- **2025:** se mezclan dos causas. Hay extremos de UxP y LLA con muchos votos
  (Valdés, Allende, Marín), y también diputados que asumieron en diciembre y
  tienen solo 17 o 18 votos útiles (Goitia, Grabois, Cámpora, Gruber): menos
  votos, más incertidumbre.
- **2026:** casi todo el top es de La Libertad Avanza, en el extremo derecho:
  ahora el bloque más disciplinado es el oficialismo.

Los desvíos no se comparan bien entre años: dependen de cuántas votaciones
hubo (en 2023 la mediana es ≈ 0.4 con unos 30 votos útiles por diputado; en
2024, ≈ 0.14 con más de 150).

## Dot plot (Wilkinson): distribución por bloque



In [ ]:
ANCHO_BIN = 0.06

dd = df.dropna(subset=["bloque"])

In [ ]:
def apilar(valores, ancho_bin):
    """Wilkinson: redondea cada valor al bin mas cercano y devuelve, para cada
    punto, en que nivel de la pila de ese bin le toca (0, 1, 2, ...)."""
    binned = (valores / ancho_bin).round() * ancho_bin
    nivel = binned.groupby(binned).cumcount()
    return binned, nivel


def mayor_hueco(g):
    gaps = g["idp_bayes"].diff()
    i_max = gaps.idxmax()
    antes, despues = g.loc[i_max - 1], g.loc[i_max]
    return gaps.loc[i_max], (antes["idp_bayes"] + despues["idp_bayes"]) / 2, antes["nombre"], despues["nombre"]


dots, huecos = {}, {}
for anio, g in dd.groupby("anio"):
    g = g.sort_values("idp_bayes").reset_index(drop=True)
    g["x_bin"], g["nivel"] = apilar(g["idp_bayes"], ANCHO_BIN)
    dots[anio], huecos[anio] = g, mayor_hueco(g)
    hueco, _, izq, der = huecos[anio]
    print(f"{anio}: mayor hueco {hueco:.2f}, entre {izq} y {der}")

In [ ]:
fig2 = go.Figure()

trazas_por_anio = {}
for anio, g in dots.items():
    trazas_por_anio[anio] = []
    for bloque_val, gb in g.groupby("bloque_mostrado"):
        trazas_por_anio[anio].append(len(fig2.data))
        fig2.add_trace(go.Scatter(
            x=gb["x_bin"], y=gb["nivel"],
            mode="markers",
            marker=dict(size=9, color=COLOR_BLOQUE[bloque_val], line=dict(width=0)),
            name=f"{bloque_val} ({len(gb)})",
            legendrank=1000 if bloque_val == "Otros bloques" else gb["idp_bayes"].median() * 100 + 500,
            customdata=list(zip(gb["nombre"], gb["bloque"], gb["idp_bayes"])),
            hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>idp = %{customdata[2]:.2f}<extra></extra>",
            hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
        ))


def vista_anio(anio):
    hueco, x_hueco, _, _ = huecos[anio]
    visibles = [i in trazas_por_anio[anio] for i in range(len(fig2.data))]
    layout = dict(
        shapes=[dict(type="line", x0=x_hueco, x1=x_hueco, yref="paper", y0=0, y1=1,
                     line=dict(color=MUTED, width=1, dash="dot"))],
        annotations=[dict(x=x_hueco, y=0.85, yref="paper", text=f"mayor hueco: {hueco:.2f}", showarrow=False,
                          font=dict(size=11, color=TEXT2), xanchor="left", xshift=6)],
    )
    return visibles, layout


inicial = anios[-1]
visibles, layout_inicial = vista_anio(inicial)
for traza, visible in zip(fig2.data, visibles):
    traza.visible = visible

botones = [dict(label=str(a), method="update", args=[{"visible": vista_anio(a)[0]}, vista_anio(a)[1]])
           for a in anios]

x_max = math.ceil(dd["idp_bayes"].abs().max() * 10) / 10 + ANCHO_BIN
fig2.update_xaxes(range=[-x_max, x_max], zeroline=True, zerolinecolor=BASELINE, zerolinewidth=1,
                   gridcolor=GRID, tickfont=dict(size=11, color=MUTED))
fig2.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)

fig2.update_layout(
    **layout_inicial,
    title=dict(text="Puntos ideales (Bayes) por bloque \u00b7 C\u00e1mara de Diputados, 2022\u20132026",
               x=0.01, y=0.97, yanchor="top", font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=520, width=1000,
    margin=dict(l=40, r=20, t=110, b=40),
    legend=dict(orientation="h", y=-0.12, x=0),
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, active=anios.index(inicial),
                      x=0.01, xanchor="left", y=1.04, yanchor="bottom", showactive=True,
                      pad=dict(r=4, t=0), bgcolor=SURFACE, bordercolor=GRID,
                      font=dict(size=11, color=TEXT2))],
)

fig2.show(config={"displaylogo": False, "scrollZoom": False})

**Qué significa el hueco de 0.71 entre Giordano y Grabois (2026).** El hueco no
separa a dos personas sino a dos grupos: a la izquierda quedan solo los cinco
diputados del Frente de Izquierda (Del Plá, Bregman, Pitrola, Del Caño y
Giordano, el más "moderado" de ellos), y a la derecha los otros 252, empezando
por Grabois, Máximo Kirchner y Taiana, que encabezan Unión por la Patria. Pero
no es la brecha política más grande de la Cámara: Giordano y Grabois votan igual
en el 95% de las votaciones que comparten. De las 107 votaciones de 2026, el
FIT se separa de UxP en solo 6, y son todas del mismo tipo: tratados y acuerdos
internacionales (Mercosur–Unión Europea, Mercosur–Singapur, extradición con
Chile, traslado de condenados con Italia, cooperación jurídica, convenio con
Francia), que el FIT vota en contra prácticamente solo, frente a toda la Cámara.

Eso alcanza para el modelo: en cada una de esas votaciones el corte cae entre el
FIT y todos los demás, y como del otro lado del FIT no hay nadie que lo
"sostenga", el modelo lo empuja hacia afuera (el mismo efecto de borde que
explica los desvíos grandes en la tabla de posiciones de más arriba). El hueco
mide, entonces, una diferencia **consistente pero acotada**: el FIT se distingue
de UxP en un tema puntual (el rechazo a los acuerdos de libre comercio y a los
tratados con otros gobiernos), no en todo. La división principal de la Cámara,
entre UxP (mediana −0.84) y LLA (mediana +0.99), es bastante más grande, pero no
aparece como hueco porque el espacio entre los dos está ocupado por Provincias
Unidas, el PRO, la UCR y los bloques provinciales. Lo mismo pasa en otros años:
el hueco más grande suele aislar a un grupo chico de un extremo (en 2024, cinco
diputados a la izquierda de todos), más que partir la Cámara en dos.

## Mapa de proyectos: corte, discriminación y dirección

El modelo de Clinton, Jackman y Rivers no solo ubica legisladores: cada votación
*j* también tiene sus parámetros, en `P(Sí) = Φ(β_j · θ_i − α_j)`:

- **Corte** `α_j / β_j`: el punto del eje izquierda–derecha donde un legislador
  queda indiferente entre Sí y No. Cerca de 0 es una votación que parte la Cámara
  al medio; lejos de 0 es una votación donde solo un extremo quedó del otro lado
  (qué tan *extrema* es).
- **Discriminación** `|β_j|`: qué tan nítido es ese corte. Alta = el voto se
  explica casi del todo por la ideología (izquierda de un lado, derecha del
  otro); cerca de 0 = el eje izquierda–derecha no explica el voto (otros clivajes:
  territoriales, de conciencia, etc.).
- **Dirección** (color): el `idp_bayes` promedio de los que votaron **Sí**. Rojo =
  la apoyó la izquierda; azul = la apoyó la derecha. Es otra forma de leer el
  signo de β, en unidades de punto ideal.

`ideal_pscl.py` muestrea β y α pero no los guarda, así que acá se reestiman sin
volver a correr la cadena: para cada votación, un probit de los votos (Sí = 1,
No = 0, el resto afuera) contra los `idp_bayes` ya estimados, con el mismo prior
N(0, 25) del modelo (estimación MAP). El prior es también el que pone techo a β
en las votaciones con separación perfecta (≈ 11). Las votaciones unánimes quedan
afuera, igual que en `preparar_pscl()`.

In [ ]:
info = pd.read_csv("datos/info_sesiones_dt.csv")

votos = (ses.assign(y=ses["voto"].map({"AFIRMATIVO": 1.0, "NEGATIVO": 0.0}))
            .dropna(subset=["y"])
            .merge(df[["nombre", "anio", "idp_bayes"]], left_on=["legislador", "yr"], right_on=["nombre", "anio"]))


def parametros_votacion(g):
    x, y = g["idp_bayes"].to_numpy(), g["y"].to_numpy()
    if y.min() == y.max():
        return None
    s = 2 * y - 1
    menos_log_post = lambda p: -log_ndtr(s * (p[0] * x - p[1])).sum() + 0.02 * (p ** 2).sum()
    beta, alpha = minimize(menos_log_post, [0.0, 0.0], method="BFGS").x
    return pd.Series({"beta": beta, "alpha": alpha, "corte": alpha / beta,
                      "discriminacion": abs(beta), "idp_si": x[y == 1].mean(),
                      "afirmativos": int(y.sum()), "negativos": int(len(y) - y.sum())})


proyectos = (votos.groupby(["anio", "id_votacion"]).apply(parametros_votacion).dropna().reset_index()
                  .merge(info[["id_votacion", "titulo", "fecha", "resultado"]], on="id_votacion"))
proyectos.describe().round(2)

In [ ]:
RANGO_CORTE = 2.5

visibles = proyectos[proyectos["corte"].abs() <= RANGO_CORTE]
print(f"{len(proyectos) - len(visibles)} de {len(proyectos)} votaciones con |corte| > {RANGO_CORTE} quedan fuera "
      f"del gráfico (casi unánimes, con discriminación baja: el corte no está bien definido)")

fig3 = go.Figure()
for anio in anios:
    g = visibles[visibles["anio"] == anio]
    fig3.add_trace(go.Scatter(
        x=g["corte"], y=g["discriminacion"], mode="markers", name=str(anio),
        marker=dict(size=9, color=g["idp_si"], colorscale=[[0, IZQUIERDA], [0.5, GRID], [1, DERECHA]],
                    cmin=-1, cmax=1, cmid=0, line=dict(width=1, color=SURFACE),
                    colorbar=dict(title=dict(text="idp medio<br>de los Sí", font=dict(size=11, color=TEXT2)),
                                  tickvals=[-1, 0, 1], ticktext=["izquierda", "centro", "derecha"],
                                  tickfont=dict(size=10, color=MUTED), thickness=12, len=0.6)),
        customdata=list(zip(g["titulo"], g["fecha"], g["resultado"], g["idp_si"], g["afirmativos"], g["negativos"])),
        hovertemplate=("<b>%{customdata[0]}</b><br>%{customdata[1]} · %{customdata[2]} "
                       "(%{customdata[4]:.0f} Sí / %{customdata[5]:.0f} No)<br>"
                       "corte = %{x:.2f} · discriminación = %{y:.2f} · idp medio Sí = %{customdata[3]:.2f}<extra></extra>"),
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
    ))

botones = [dict(label="Todos", method="update", args=[{"visible": [True] * len(anios)}])]
botones += [dict(label=str(a), method="update", args=[{"visible": [b == a for b in anios]}]) for a in anios]

fig3.update_xaxes(title=dict(text="corte (α/β) · ← izquierda · derecha →", font=dict(size=12, color=TEXT2)),
                  range=[-RANGO_CORTE, RANGO_CORTE], zeroline=True, zerolinecolor=BASELINE,
                  gridcolor=GRID, tickfont=dict(size=11, color=MUTED))
fig3.update_yaxes(title=dict(text="discriminación |β|", font=dict(size=12, color=TEXT2)),
                  rangemode="tozero", gridcolor=GRID, zeroline=False, tickfont=dict(size=11, color=MUTED))
fig3.update_layout(
    title=dict(text="Votaciones 2022–2026: dónde cortan, qué tan nítido y quién las apoyó",
               x=0.01, font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=600, width=1000, showlegend=False,
    margin=dict(l=60, r=20, t=110, b=60),
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, x=0.01, xanchor="left",
                      y=1.02, yanchor="bottom", showactive=True, pad=dict(r=4, t=0),
                      bgcolor=SURFACE, bordercolor=GRID, font=dict(size=11, color=TEXT2))],
)
fig3.show(config={"displaylogo": False, "scrollZoom": False})




**Cómo leer un punto, paso a paso** (cada punto es una votación, no un diputado):

1. **Altura**: ¿la votación fue ideológica? Arriba, sí (el voto se predice con el
   punto ideal); abajo, no.
2. **Posición horizontal**: ¿partió la Cámara al medio (centro) o dejó del otro
   lado solo a un extremo (costados)?
3. **Color**: ¿quién la impulsó? Rojo, la izquierda; azul, la derecha; gris, casi
   todos.

Cuatro ejemplos, uno por zona:

- **Arriba al centro: grieta pura.** Los artículos de la Ley Bases (2024): corte
  ≈ 0.09, discriminación ≈ 11.7, 134 a 119. La Cámara partida justo al medio y
  casi nadie votó distinto de lo que predecía su posición. Azul: la apoyó la
  derecha (idp medio de los Sí = +0.82).
- **Arriba a la izquierda: solo la punta izquierda votó distinto.** Convenio con
  Austria para eliminar la doble imposición (2024): 224 a 5, corte −1.8. Los 5 No
  son el Frente de Izquierda; todo el resto votó Sí, por eso el color es gris.
- **Arriba a la derecha: solo la punta derecha votó distinto.** Ley de Respuesta
  Integral al VIH, Hepatitis Virales, ITS y TBC (2022): 241 a 8, corte +1.53. Los
  8 No están en el extremo derecho.
- **Abajo: el eje izquierda–derecha no explica el voto.** Emergencia en
  discapacidad, nuevo artículo 2 (2025): 123 a 82, pero con discriminación de
  0.21: hubo Sí y No en todo el espectro. Pesaron otras cosas (provincias,
  negociaciones puntuales), no la ideología.

## Grafos de cercanía y comunidades

Lo mismo que `04_redes.py` (el script de redes de la materia), pero dibujado acá
con plotly, con una solapa por año. Se parte de `datos/distancias_idps.csv.gz`: el producto
cartesiano de diputados de cada año, con la distancia entre sus puntos ideales
(`|idp_bayes A - idp_bayes B|`).

Para cada año:

1. Se arma la red completa: un nodo por diputado y una arista por par, con la
   distancia como peso. Sobre esa red se calcula la **intermediación**
   (betweenness) de cada diputado.
2. Se **poda**: se borran las aristas con distancia mayor a `PUNTO_CORTE` (0.4),
   y después los diputados que quedan con menos de 3 uniones.
3. Se detectan **comunidades** con fast greedy (peso `1 / distancia`, así una
   unión corta pesa más), cortando donde la modularidad es máxima.

En el dibujo, cada comunidad (las 8 más grandes; el resto en gris) toma el
color de su bloque mayoritario, el mismo que en el resto del trabajo; el
tamaño del nodo es su intermediación y se etiquetan los 5 diputados con más
intermediación (pasar el mouse sobre cualquier nodo muestra nombre, bloque e
intermediación). La leyenda dice, para cada comunidad, qué bloque es mayoría y en
qué proporción. La posición sale de Fruchterman-Reingold, así que la forma
cambia un poco entre corridas (la estructura no).

In [ ]:
PUNTO_CORTE = 0.4
MAX_COMUNIDADES = 8
OTRAS, ARISTA = "#c3c2b7", "#898781"

distancias = pd.read_csv("datos/distancias_idps.csv.gz").rename(
    columns={"nombre1": "Source", "nombre2": "Target", "distancia": "weight"})
distancias = distancias[distancias.Source < distancias.Target]


def armar_grafo(anio):
    dist = distancias[distancias.anio == anio]
    dips = bloques[bloques.yr == anio]
    g = ig.Graph.DataFrame(dist[["Source", "Target", "weight"]], directed=False,
                           vertices=dips[["legislador"]], use_vids=False)
    g.vs["bloque"] = list(dips.bloque)
    return g


def podar(g):
    g = g.copy()
    g.simplify(combine_edges="sum")
    g.delete_edges(g.es.select(weight_gt=PUNTO_CORTE))
    g.simplify(combine_edges="sum")
    g.delete_vertices(g.vs.select(_degree_lt=3))
    return g


def intermediacion(g):
    n = g.vcount()
    return [b * 2 / ((n - 1) * (n - 2)) for b in g.betweenness(directed=False, weights="weight")]


def comunidades(g):
    arbol = g.community_fastgreedy(weights=[1 / w for w in g.es["weight"]])
    membresia = ig.community_to_membership(arbol.merges, g.vcount(), g.vcount() - arbol.optimal_count)
    return ig.VertexClustering(g, membresia)

In [ ]:
def calcular_red(anio):
    g = armar_grafo(anio)
    g.vs["betweenness"] = intermediacion(g)
    g = podar(g)
    x = comunidades(g)
    por_tamanio = sorted(range(len(x)), key=lambda c: -len(x[c]))
    random.seed(int(anio))  # el layout arranca de posiciones al azar: con la semilla, el dibujo sale igual en cada corrida
    xs, ys = zip(*g.layout_fruchterman_reingold(weights=[1 / w for w in g.es["weight"]], niter=2000).coords)
    bloque = ["Frente de Izquierda" if "izquierda" in b.lower() else NOMBRE_BLOQUE.get(b, b) for b in g.vs["bloque"]]
    nodos = pd.DataFrame({"legislador": g.vs["name"],
                          "nombre": [n.replace('"', "").title() for n in g.vs["name"]], "bloque": bloque,
                          "betweenness": g.vs["betweenness"], "x": xs, "y": ys,
                          "comunidad": [por_tamanio.index(m) for m in x.membership]})
    aristas = [e.tuple for e in g.es]
    modularidad = g.modularity(x.membership, weights=[1 / w for w in g.es["weight"]])
    return nodos, aristas, len(x), modularidad


redes = {anio: calcular_red(anio) for anio in anios}

En 2025 hay más diputados que bancas (257) porque el 10 de diciembre cambió la
composición de la Cámara: en `legisladores_bloque_provincia.csv` aparecen tanto
los que salieron como los que entraron ese año.

In [ ]:
fig4 = go.Figure()
trazas_red = {}
titulos_red = {}
etiquetas_red = {}

for anio, (nodos, aristas, n_comunidades, _) in redes.items():
    trazas_red[anio] = []
    titulos_red[anio] = (f"Grafo de cercan\u00eda \u00b7 {anio} \u00b7 {len(nodos)} diputados, "
                         f"{len(aristas)} uniones, {n_comunidades} comunidades")

    ex, ey = [], []
    for a, b in aristas:
        ex += [nodos.x[a], nodos.x[b], None]
        ey += [nodos.y[a], nodos.y[b], None]
    trazas_red[anio].append(len(fig4.data))
    fig4.add_trace(go.Scatter(x=ex, y=ey, mode="lines", line=dict(color="rgba(137, 135, 129, 0.06)", width=0.5),
                                hoverinfo="skip", showlegend=False))

    nodos = nodos.assign(grupo=nodos["comunidad"].clip(upper=MAX_COMUNIDADES))
    for grupo, gn in nodos.groupby("grupo"):
        if grupo < MAX_COMUNIDADES:
            bloque = gn["bloque"].mode()[0]
            color = COLOR_BLOQUE.get(bloque, OTRAS)
            nombre = f"{len(gn)} dip. \u00b7 {(gn['bloque'] == bloque).mean():.0%} {bloque}"
        else:
            color = OTRAS
            nombre = f"{len(gn)} dip. \u00b7 otras {n_comunidades - MAX_COMUNIDADES} comunidades"
        trazas_red[anio].append(len(fig4.data))
        fig4.add_trace(go.Scatter(
            x=gn["x"], y=gn["y"], mode="markers", name=nombre,
            marker=dict(size=(10 + 500 * gn["betweenness"]) ** 0.5 * 1.3, color=color,
                        line=dict(width=0.6, color=SURFACE)),
            customdata=list(zip(gn["nombre"], gn["bloque"], gn["betweenness"])),
            hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>intermediaci\u00f3n = %{customdata[2]:.3f}<extra></extra>",
            hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
        ))

    top = nodos.nlargest(5, "betweenness").sort_values("x")
    etiquetas_red[anio] = [dict(x=n.x, y=n.y, text=n.nombre, showarrow=True, arrowhead=0, arrowwidth=0.6,
                                arrowcolor=MUTED, ax=0, ay=-40 - 28 * k, font=dict(size=11, color=TEXT),
                                bgcolor=SURFACE, borderpad=1)
                           for k, n in enumerate(top.itertuples())]


def vista_red(anio):
    visibles = [i in trazas_red[anio] for i in range(len(fig4.data))]
    return [{"visible": visibles}, {"title.text": titulos_red[anio], "annotations": etiquetas_red[anio]}]


inicial = anios[-1]
for traza, visible in zip(fig4.data, vista_red(inicial)[0]["visible"]):
    traza.visible = visible

botones = [dict(label=str(a), method="update", args=vista_red(a)) for a in anios]

fig4.update_xaxes(visible=False)
fig4.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)
fig4.update_layout(
    annotations=etiquetas_red[inicial],
    title=dict(text=titulos_red[inicial], x=0.01, y=0.97, yanchor="top",
               font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=750, width=1100,
    margin=dict(l=20, r=20, t=110, b=20),
    legend=dict(itemsizing="constant", title=dict(text="Comunidad \u00b7 bloque mayoritario", font=dict(size=11, color=TEXT2)),
                x=1.0, y=1.0, xanchor="left", yanchor="top", font=dict(size=11, color=TEXT2)),
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, active=anios.index(inicial),
                      x=0.01, xanchor="left", y=1.04, yanchor="bottom", showactive=True,
                      pad=dict(r=4, t=0), bgcolor=SURFACE, bordercolor=GRID,
                      font=dict(size=11, color=TEXT2))],
)

fig4.show(config={"displaylogo": False, "scrollZoom": False})

## Comunidades y bloques

Las comunidades del grafo no coinciden con los bloques. Acá las cruzamos, año
por año, para responder tres preguntas: **¿qué bloque está partido?**, **¿quién
vota lejos de su bloque?** y **¿qué cambió?**

Decisiones:

- **Familias de bloques.** Para comparar entre años hacen falta categorías que
  no cambien. Agrupamos los bloques en seis familias: Izquierda (todos los
  bloques del Frente de Izquierda), FdT / UxP (Frente de Todos y su sucesor,
  Unión por la Patria), PRO, UCR, LLA y Otros. "Otros" junta bloques muy
  distintos entre sí (Provincias Unidas, Encuentro Federal, Innovación Federal,
  etc.): sirve como destino pero no como identidad.
- **Comunidades numeradas por el eje.** C1 es la comunidad con el punto ideal
  medio más a la izquierda (en el grafo de arriba se ordenan por tamaño). Como
  la red sale de distancias sobre un solo eje, cada comunidad es, en la
  práctica, un tramo de ese eje, cortado donde hay pocos diputados. Por eso
  cada una se rotula con su rango de puntos ideales. Para eso se usa el punto
  ideal con el que se armó la red (el de `distancias_idps.csv.gz`), que es casi
  igual a nuestro `idp_bayes` (correlación 0.99) pero no idéntico: con
  `idp_bayes` los tramos se superponen un poco.

In [ ]:
FAMILIA = {"Frente de Izquierda": "Izquierda", "Frente de Todos": "FdT / UxP", "Unión por la Patria": "FdT / UxP",
           "PRO": "PRO", "UCR": "UCR", "La Libertad Avanza": "LLA"}
FAMILIAS = ["Izquierda", "FdT / UxP", "Otros", "PRO", "UCR", "LLA"]
COLOR_FAMILIA = {FAMILIA[b]: COLOR_BLOQUE[b] for b in FAMILIA} | {"Otros": OTRAS}
MIN_POR_COMUNIDAD = 3


# El punto ideal con el que se armó la red (viene en distancias_idps.csv.gz). Es casi igual a nuestro
# idp_bayes (correlación 0.99 cada año), pero no idéntico: para que cada comunidad sea un tramo
# limpio del eje hay que ordenarlas con el mismo punto ideal que definió las uniones.
idp_red = pd.concat([distancias[["Source", "anio", "idp_desde"]].set_axis(["legislador", "anio", "idp"], axis=1),
                     distancias[["Target", "anio", "idp_hasta"]].set_axis(["legislador", "anio", "idp"], axis=1)]
                    ).drop_duplicates(["legislador", "anio"])


def comunidades_por_eje(anio):
    """Los nodos de la red de un año, con su punto ideal, su familia y su comunidad
    renumerada de izquierda a derecha según el punto ideal medio (C1 es la de más a la izquierda)."""
    nodos = (redes[anio][0]
             .merge(idp_red.loc[idp_red["anio"] == anio, ["legislador", "idp"]], on="legislador", how="left",
                    validate="one_to_one")
             .merge(df.loc[df["anio"] == anio, ["nombre", "votos_utiles"]].rename(columns={"nombre": "legislador"}),
                    on="legislador", how="left", validate="one_to_one"))
    nodos["familia"] = nodos["bloque"].map(FAMILIA).fillna("Otros")
    orden = nodos.groupby("comunidad")["idp"].mean().rank(method="first").astype(int)
    nodos["com_eje"] = nodos["comunidad"].map(orden)
    mayoria = nodos.groupby("com_eje")["familia"].agg(lambda s: s.value_counts().index[0])
    nodos["familia_comunidad"] = nodos["com_eje"].map(mayoria)
    return nodos


def resumen_comunidades(nodos):
    r = nodos.groupby("com_eje").agg(n=("legislador", "size"), desde=("idp", "min"), hasta=("idp", "max"),
                                     familia=("familia_comunidad", "first"))
    r["etiqueta"] = [f"C{c} · {d:+.2f} a {h:+.2f} · {n} dip." for c, n, d, h in zip(r.index, r.n, r.desde, r.hasta)]
    return r


def menu_anios(vista, opciones, inicial):
    """Botones para cambiar de año, con el mismo estilo que el grafo de arriba."""
    return [dict(type="buttons", direction="right", buttons=[dict(label=str(a), method="update", args=vista(a))
                                                             for a in opciones],
                 active=opciones.index(inicial), x=0.01, xanchor="left", y=1.04, yanchor="bottom", showactive=True,
                 pad=dict(r=4, t=0), bgcolor=SURFACE, bordercolor=GRID, font=dict(size=11, color=TEXT2))]


def texto_sobre(color):
    """Texto oscuro sobre colores claros y claro sobre colores oscuros."""
    r, g, b = (int(color[i:i + 2], 16) for i in (1, 3, 5))
    return TEXT if 0.299 * r + 0.587 * g + 0.114 * b > 150 else SURFACE


nodos_anio = {anio: comunidades_por_eje(anio) for anio in anios}
resumen_anio = {anio: resumen_comunidades(n) for anio, n in nodos_anio.items()}
resumen_anio[anios[-1]]

### ¿Qué bloque está partido?

Cada fila es una comunidad, de izquierda a derecha del eje; cada columna, una
familia de bloques. El número es la cantidad de diputados y el color, qué parte
de la comunidad representan. Una familia está **partida** cuando sus diputados
aparecen en varias filas.

In [ ]:
fig6 = go.Figure()
ejes_cruce = {}
for anio in anios:
    nodos, r = nodos_anio[anio], resumen_anio[anio]
    cuenta = pd.crosstab(nodos["com_eje"], nodos["familia"]).reindex(index=r.index, columns=FAMILIAS, fill_value=0)
    pct = cuenta.div(cuenta.sum(axis=1), axis=0)
    hover = [[f"<b>{r.etiqueta.iat[i]}</b><br>{f}: {cuenta.iat[i, j]} ({pct.iat[i, j]:.0%} de la comunidad)"
              for j, f in enumerate(FAMILIAS)] for i in range(len(r))]
    fig6.add_trace(go.Heatmap(
        z=pct.to_numpy(), x=list(range(len(FAMILIAS))), y=list(range(len(r))),
        text=cuenta.where(cuenta > 0, "").to_numpy(), texttemplate="%{text}", textfont=dict(size=12, color=TEXT),
        customdata=hover, hovertemplate="%{customdata}<extra></extra>",
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
        colorscale=[[0, SURFACE], [1, "#8fa3b8"]], zmin=0, zmax=1, showscale=False, xgap=2, ygap=2,
        visible=anio == anios[-1]))
    ejes_cruce[anio] = dict(tickvals=list(range(len(r))), ticktext=list(r["etiqueta"]))


def vista_cruce(anio):
    return [{"visible": [a == anio for a in anios]},
            {"title.text": f"Comunidades × familias de bloques · {anio}",
             "yaxis.tickvals": ejes_cruce[anio]["tickvals"], "yaxis.ticktext": ejes_cruce[anio]["ticktext"]}]


inicial = anios[-1]
fig6.update_xaxes(side="top", tickvals=list(range(len(FAMILIAS))), showgrid=False, zeroline=False,
                  ticktext=[f"<span style='color:{COLOR_FAMILIA[f]}'>●</span> {f}" for f in FAMILIAS])
fig6.update_yaxes(autorange="reversed", showgrid=False, zeroline=False, **ejes_cruce[inicial])
fig6.update_layout(
    title=dict(text=vista_cruce(inicial)[1]["title.text"], x=0.01, y=0.97, yanchor="top",
               font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT, size=11), paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=600, width=1000, margin=dict(l=20, r=20, t=140, b=20),
    updatemenus=menu_anios(vista_cruce, anios, inicial),
)
fig6.show(config={"displaylogo": False})

In [ ]:
FAMILIAS_PROPIAS = [f for f in FAMILIAS if f != "Otros"]

partido, detalle = {}, {}
for anio, nodos in nodos_anio.items():
    for f in FAMILIAS_PROPIAS:
        por_com = nodos.loc[nodos["familia"] == f, "com_eje"].value_counts().sort_index()
        if por_com.sum() < MIN_POR_COMUNIDAD:
            continue
        partido[f, anio] = (por_com >= MIN_POR_COMUNIDAD).sum()
        detalle[f, anio] = " · ".join(f"C{c}: {n}" for c, n in por_com.items())

fig7 = make_subplots(rows=1, cols=len(FAMILIAS_PROPIAS), subplot_titles=FAMILIAS_PROPIAS, shared_yaxes=True,
                     horizontal_spacing=0.03)
for k, f in enumerate(FAMILIAS_PROPIAS, start=1):
    x = [a for a in anios if (f, a) in partido]
    fig7.add_trace(go.Bar(
        x=[str(a) for a in x], y=[partido[f, a] for a in x], marker=dict(color=COLOR_FAMILIA[f], line=dict(width=0)),
        text=[partido[f, a] for a in x], textposition="outside", textfont=dict(size=11, color=TEXT2),
        customdata=[detalle[f, a] for a in x],
        hovertemplate=f"<b>{f} · %{{x}}</b><br>diputados por comunidad: %{{customdata}}<extra></extra>",
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12), showlegend=False,
    ), row=1, col=k)
fig7.update_xaxes(type="category", categoryorder="array", categoryarray=[str(a) for a in anios],
                  tickfont=dict(size=10, color=TEXT2), showgrid=False)
fig7.update_yaxes(showgrid=True, gridcolor=GRID, zeroline=False, tickfont=dict(size=10, color=TEXT2),
                  range=[0, max(partido.values()) + 1.5])
fig7.update_annotations(font=dict(size=13, color=TEXT))
fig7.update_layout(
    title=dict(text=f"¿Qué bloque está partido? · comunidades con {MIN_POR_COMUNIDAD} o más diputados de cada familia",
               x=0.01, font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT), paper_bgcolor=PAGE, plot_bgcolor=SURFACE, bargap=0.25,
    height=380, width=1100, margin=dict(l=40, r=20, t=90, b=40),
)
fig7.show(config={"displaylogo": False})

**El peronismo está partido todos los años; el más compacto es el de 2026.**
FdT / UxP ocupa entre 4 y 6 comunidades de 2022 a 2025 y baja a 3 en 2026: tras
la renovación de diciembre de 2025 queda un bloque más chico (92 diputados) y
más ordenado. Aun así sigue en tres tramos bien separados.

**PRO y UCR se parten en los años de transición.** El PRO ocupa 4 comunidades en
2023 (año electoral) y en 2025 (año en que conviven dos composiciones de la
Cámara), y una sola en 2024 y 2026. La UCR pasa de 4 comunidades en 2022–2023
a 1 desde 2024; parte de esa baja es mecánica, porque el bloque se achica
(de 33 diputados en 2022 a 6 en 2026).

**LLA en 3 o 4 comunidades no es una interna.** En 2026 sus tres comunidades
(C8 a C10) votan igual: en las 107 votaciones divididas del año, ninguno de
los 102 diputados de esas comunidades vota distinto de la mayoría. En la punta
del eje el modelo no llega a distinguirlos, y el corte las separa igual.
Que una familia ocupe varias comunidades indica una división solo si hay
votaciones que la expliquen.

### ¿Quién vota lejos de su bloque?

Un diputado **vota lejos de su bloque** cuando cae en una comunidad cuya
mayoría es de otra familia: el bloque dice con quién se sienta y la comunidad,
con quién vota. Se dejan afuera los de "Otros", que no tienen una familia propia,
y los que tienen menos de `MIN_VOTOS_UTILES` votos útiles ese año (ver la tabla
de desvíos), para no contar como desplazados a los que el modelo ubica mal por
falta de votos. Cada segmento de la barra es un flujo "familia → familia de su
comunidad", con el color de la familia de origen.

In [ ]:
fuera = pd.concat([n.assign(anio=anio) for anio, n in nodos_anio.items()])
fuera = fuera[(fuera["familia"] != "Otros") & (fuera["familia"] != fuera["familia_comunidad"])
              & (fuera["votos_utiles"] >= MIN_VOTOS_UTILES)].copy()
fuera["flujo"] = fuera["familia"] + " → " + fuera["familia_comunidad"]
flujos = (fuera.groupby(["familia", "flujo", "anio"])["nombre"]
          .agg(n="size", nombres=lambda s: "<br>".join(sorted(s)[:12]) + ("<br>…" if len(s) > 12 else ""))
          .reset_index())

fig8 = go.Figure()
for flujo, gf in flujos.groupby("flujo", sort=False):
    color = COLOR_FAMILIA[gf["familia"].iat[0]]
    fig8.add_trace(go.Bar(
        x=gf["anio"].astype(str), y=gf["n"], name=flujo,
        marker=dict(color=color, line=dict(width=1.5, color=SURFACE)),
        text=[f"{flujo} · {n}" if n >= 4 else "" for n in gf["n"]], textposition="inside",
        insidetextanchor="middle", textfont=dict(size=11, color=texto_sobre(color)),
        customdata=gf["nombres"],
        hovertemplate=f"<b>{flujo} · %{{x}}</b> · %{{y}} diputados<br>%{{customdata}}<extra></extra>",
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12), showlegend=False,
    ))
totales = flujos.groupby("anio")["n"].sum()
for anio, n in totales.items():
    fig8.add_annotation(x=anios.index(anio), y=n, text=f"<b>{n}</b>", showarrow=False, yshift=10,
                        font=dict(size=12, color=TEXT))
fig8.update_xaxes(type="category", categoryorder="array", categoryarray=[str(a) for a in anios], showgrid=False)
fig8.update_yaxes(title=dict(text="diputados", font=dict(size=11, color=TEXT2)), gridcolor=GRID, zeroline=False)
fig8.update_layout(
    barmode="stack", bargap=0.2,
    title=dict(text="¿Quién vota lejos de su bloque? · diputados en una comunidad de mayoría de otra familia",
               x=0.01, font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT), paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=560, width=1100, margin=dict(l=60, r=20, t=70, b=40),
)
fig8.show(config={"displaylogo": False})

In [ ]:
fig9 = go.Figure()
for anio in anios:
    g = fuera[fuera["anio"] == anio].sort_values(["flujo", "idp"])
    fig9.add_trace(go.Table(
        columnwidth=[1.4, 1.4, 1.2, 0.5, 0.6],
        header=dict(values=["<b>Diputado</b>", "<b>Bloque</b>", "<b>Vota con</b>", "<b>idp</b>", "<b>votos útiles</b>"],
                    fill_color=PAGE, line_color=GRID, font=dict(size=12, color=TEXT), align="left", height=30),
        cells=dict(values=[g["nombre"], g["bloque"], [f"C{c} · mayoría {f}" for c, f in zip(g["com_eje"], g["familia_comunidad"])],
                           [f"{i:+.2f}" for i in g["idp"]], g["votos_utiles"].astype(int)],
                   fill_color=[[suave(COLOR_FAMILIA[f]) for f in g["familia"]]] * 5, line_color=SURFACE, align="left",
                   height=26, font=dict(size=11, color=TEXT)),
        visible=anio == anios[-1]))


def vista_fuera(anio):
    return [{"visible": [a == anio for a in anios]},
            {"title.text": f"Los que votan lejos de su bloque · {anio} · {(fuera['anio'] == anio).sum()} diputados"}]


fig9.update_layout(
    title=dict(text=vista_fuera(inicial)[1]["title.text"], x=0.01, y=0.97, yanchor="top",
               font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT), paper_bgcolor=PAGE,
    height=600, width=1000, margin=dict(l=20, r=20, t=110, b=20),
    updatemenus=menu_anios(vista_fuera, anios, inicial),
)
fig9.show(config={"displaylogo": False})

**Cada año tiene su flujo dominante, y cuenta la historia de las alianzas.**

- **2022: la izquierda "con" la UCR (4).** Es el efecto ya explicado más arriba:
  en 2022 el eje mide cercanía al gobierno y el Frente de Izquierda queda del
  lado de la oposición de entonces.
- **2023: la UCR vota con el PRO (16).** Es Juntos por el Cambio funcionando como
  una sola oposición.
- **2024: la UCR se va al centro (21).** Con el cambio de gobierno, los radicales
  dejan de votar con el PRO y caen en la comunidad del medio, junto a los
  bloques de gobernadores.
- **2025: el PRO vota con LLA (26).** El flujo más grande de los cinco años: el
  PRO queda adentro de comunidades de La Libertad Avanza. En el otro extremo,
  la izquierda vota con Unión por la Patria (11) como parte de la oposición.
- **2026: casi nadie vota lejos de su bloque (9).** Los que quedan son sobre todo
  radicales dentro de comunidades de LLA (6).

Cuidado: "votar lejos de su bloque" se mide sobre un solo eje. Un diputado
puede caer en la comunidad de otro bloque por votar parecido en lo que ordena
el eje (acompañar o no al gobierno) y diferir en todo lo demás.

### ¿Qué cambió? Las comunidades de un año al siguiente

Las cintas siguen a los diputados que están en los dos años: de qué comunidad
salen y a cuál llegan. Cada nodo es una comunidad, con el color de su familia
mayoritaria, en el orden del eje (C1 arriba). En color, los flujos de
`MIN_POR_COMUNIDAD` o más diputados; en gris, el resto. **2023 → 2024** es el
cambio de gobierno; **2025 → 2026**, la nueva Cámara, para comparar. La comparación es por pertenencia a comunidades y no por
posición en el eje, así que no la afecta que la escala del punto ideal se fije
año por año.

In [ ]:
TRANSICIONES = [(2023, 2024), (2025, 2026)]
NOMBRE_TRANSICION = {t: f"{t[0]} → {t[1]}" for t in TRANSICIONES}


def sankey(a1, a2):
    n1, n2 = (nodos_anio[a].set_index("legislador") for a in (a1, a2))
    comunes = n1.index.intersection(n2.index)
    cruce = pd.crosstab(n1.loc[comunes, "com_eje"], n2.loc[comunes, "com_eje"])
    r1, r2 = resumen_anio[a1], resumen_anio[a2]
    lados = [(a1, r1, cruce.sum(axis=1), 0.001), (a2, r2, cruce.sum(axis=0), 0.999)]
    etiquetas, colores, xs, ys = [], [], [], []
    for anio, r, presentes, x in lados:
        presentes = presentes.reindex(r.index, fill_value=0)
        acumulado = presentes.cumsum() / presentes.sum()
        for c in r.index:
            etiquetas.append(f"C{c} · {r.familia[c]} · {presentes[c]} de {r.n[c]}")
            colores.append(COLOR_FAMILIA[r.familia[c]])
            xs.append(x)
            ys.append(0.02 + 0.96 * (acumulado[c] - presentes[c] / presentes.sum() / 2))
    origen, destino, valor, color = [], [], [], []
    for (c1, c2), v in cruce.stack().items():
        if v == 0:
            continue
        origen.append(r1.index.get_loc(c1))
        destino.append(len(r1) + r2.index.get_loc(c2))
        valor.append(v)
        color.append(suave(COLOR_FAMILIA[r1.familia[c1]], 0.45) if v >= MIN_POR_COMUNIDAD else suave(OTRAS, 0.3))
    traza = go.Sankey(
        arrangement="snap",
        node=dict(label=etiquetas, color=colores, x=xs, y=ys, pad=8, thickness=14, line=dict(width=0),
                  hovertemplate="%{label}<extra></extra>"),
        link=dict(source=origen, target=destino, value=valor, color=color,
                  hovertemplate="%{source.label} → %{target.label}<br><b>%{value} diputados</b><extra></extra>"),
        textfont=dict(size=11, color=TEXT, family=FONT), visible=(a1, a2) == TRANSICIONES[0])
    titulo = (f"De {a1} a {a2}: cómo se reacomodaron las comunidades · "
              f"{len(comunes)} diputados presentes en los dos años")
    return traza, titulo


fig10 = go.Figure()
titulos_sankey = {}
for t in TRANSICIONES:
    traza, titulos_sankey[NOMBRE_TRANSICION[t]] = sankey(*t)
    fig10.add_trace(traza)


def vista_sankey(nombre):
    return [{"visible": [NOMBRE_TRANSICION[t] == nombre for t in TRANSICIONES]}, {"title.text": titulos_sankey[nombre]}]


opciones = list(NOMBRE_TRANSICION.values())
fig10.update_layout(
    title=dict(text=titulos_sankey[opciones[0]], x=0.01, y=0.97, yanchor="top",
               font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT), paper_bgcolor=PAGE,
    height=820, width=1100, margin=dict(l=20, r=20, t=110, b=30),
    updatemenus=menu_anios(vista_sankey, opciones, opciones[0]),
)
fig10.show(config={"displaylogo": False})

In [ ]:
filas = []
for anio, (nodos, aristas, _, modularidad) in redes.items():
    g = ig.Graph(n=len(nodos), edges=aristas)
    familia = nodos_anio[anio]["familia"]
    filas.append(dict(anio=anio, densidad=g.density(), modularidad=modularidad,
                      entre_familias=np.mean([familia.iat[a] != familia.iat[b] for a, b in aristas]),
                      asortatividad=g.assortativity_nominal(pd.factorize(familia)[0].tolist(), directed=False)))
red_anual = pd.DataFrame(filas).set_index("anio")

MEDIDAS_RED = {"densidad": "Densidad<br><sup>uniones / pares posibles</sup>",
               "modularidad": "Modularidad<br><sup>qué tan marcadas son las comunidades</sup>",
               "entre_familias": "Uniones entre familias distintas<br><sup>% de las uniones</sup>",
               "asortatividad": "Asortatividad por familia<br><sup>cuánto se une cada uno con los suyos</sup>"}

fig11 = make_subplots(rows=1, cols=len(MEDIDAS_RED), subplot_titles=list(MEDIDAS_RED.values()),
                      horizontal_spacing=0.06)
for k, medida in enumerate(MEDIDAS_RED, start=1):
    formato = "{:.0%}" if medida in ("densidad", "entre_familias") else "{:.2f}"
    fig11.add_trace(go.Scatter(
        x=red_anual.index, y=red_anual[medida], mode="lines+markers+text",
        text=[formato.format(v) for v in red_anual[medida]], textposition="top center",
        textfont=dict(size=10, color=TEXT2), line=dict(color=TEXT2, width=2), marker=dict(size=7, color=TEXT2),
        hoverinfo="skip", showlegend=False, cliponaxis=False,
    ), row=1, col=k)
    fig11.update_yaxes(tickformat=".0%" if "%" in formato else ".1f", row=1, col=k)
for x, texto in [(2023.5, "cambio de<br>gobierno"), (2025.5, "nueva<br>Cámara")]:
    fig11.add_vline(x=x, line=dict(color=BASELINE, width=1, dash="dash"))
    fig11.add_annotation(x=x, y=0.03, yref="y domain", text=texto, showarrow=False, xanchor="center", yanchor="bottom",
                         font=dict(size=10, color=MUTED), bgcolor=SURFACE, row=1, col=1)
fig11.update_xaxes(tickvals=anios, tickfont=dict(size=10, color=TEXT2), showgrid=False, range=[2021.6, 2026.4])
fig11.update_yaxes(gridcolor=GRID, zeroline=False, tickfont=dict(size=10, color=TEXT2), rangemode="tozero")
fig11.update_annotations(font=dict(size=12, color=TEXT), selector=dict(xref="paper"))
fig11.update_layout(
    title=dict(text="La red entera, año por año · corte 0.4", x=0.01, font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT), paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=380, width=1200, margin=dict(l=40, r=20, t=110, b=40),
)
fig11.show(config={"displaylogo": False})

**El cambio de gobierno reordenó el centro, no los extremos.** De 2023 a 2024
las tres comunidades del peronismo se redistribuyen entre sí, pero siguen siendo
peronistas: casi no hay cintas que crucen al resto. Lo que cambia es la
oposición de entonces: el PRO de 2023 se reparte entre la comunidad del medio
de 2024 (la más grande, con 71 diputados, donde también caen los radicales) y
una comunidad propia junto a LLA. Es decir, Juntos por el Cambio se separa en
un sector "dialoguista" y otro alineado con el nuevo gobierno. En 2025 → 2026
pasa lo contrario: el peronismo se
reacomoda más y la derecha se concentra en las comunidades de LLA.

**La Cámara de 2026 es la más alineada por bloque.** Tiene el porcentaje más
bajo de uniones entre familias distintas (19 %) y la asortatividad más alta
(0.69). La de 2025 es la más mezclada (33 % y 0.54), en parte porque ese año
conviven dos composiciones de la Cámara (y muchos diputados con pocos votos).
La densidad cae después del cambio de gobierno (de 33 % en 2023 a 26 % en 2024):
con el nuevo eje, menos diputados quedan a menos de 0.4 entre sí. La
modularidad casi no se mueve (0.75 a 0.79): en una red armada sobre un solo
eje siempre sale alta, así que no conviene leerla como "más o menos
polarización".

Advertencias: la escala del punto ideal se fija año por año, así que la
densidad y las uniones entre familias son una comparación gruesa; y las cintas siguen solo a los diputados presentes en los
dos años (149 de 2023 a 2024, por la renovación de media Cámara).

## Nube de palabras: qué proyectos apoya cada lado

Con los títulos de las votaciones del mapa de proyectos, dos nubes: las palabras
de los proyectos que apoyó la izquierda y las de los que apoyó la derecha. El
lado sale del mismo `idp_si` del mapa (el `idp_bayes` promedio de los que
votaron Sí): izquierda si es ≤ −0.4, derecha si es ≥ 0.4; lo del medio
(tratados, convenios, votaciones con apoyo amplio) queda afuera.

Antes de contar:

- Se descartan las votaciones de trámite (apartamientos de reglamento,
  mociones, habilitaciones, reconsideraciones, plan de labor, insistencias):
  sus títulos tienen nombres de diputados, no temas.
- Cada proyecto cuenta una sola vez: el Presupuesto o la Ley Bases se votaron
  decenas de veces (en general y por capítulo) y si no taparían todo lo demás.
  Las votaciones se agrupan por número de orden del día y año, y de cada
  proyecto se toman las palabras de todos sus títulos.
- Se sacan las palabras de procedimiento ("dictamen de mayoría", "votación en
  general", "modificación", "creación", números romanos de capítulos,
  abreviaturas) y las tildes al comparar (los títulos mezclan CREACION y
  CREACIÓN).

El tamaño de cada palabra es en cuántos proyectos de ese lado aparece. Hace
falta el paquete `wordcloud` (`pip install wordcloud`).

In [ ]:
UMBRAL_IDP = 0.4
TRAMITE = r"^(APARTAMIENTO|MOCI[ÓO]N|HABILITACI[ÓO]N|RECONSIDERACI[ÓO]N|PLAN DE LABOR|VOTACI[ÓO]N DESARROLLO|INSISTENCIA)"
NO_TEMA = set("""
ley leyes vot votacion gral general part particular dict dictamen may mayoria minoria modif modificacion modificaciones
mod creacion aprobacion establecimiento declaracion insistir insistencia proyecto sancion honorable camara diputados
senado hsn hcdn acep aceptacion introd introducidas nacional nacionales nac rep republica argentina argentinos art
articulo arts titulo capitulo inciso expte exptes exp del las los por para con una uno sus que entre sobre como desde
ante sin hasta otros otras diversas diversos distintos todo toda total totalidad dur susp corresp ambito marco relativas
relacionadas cuestiones redaccion texto propuesto venido revision materia nuevo nueva pens cen lib reg cap ant proy arg
inst prov desarr adm cod simpl presup imp decl transp fed preserv amb reinc reit unif derog acreed tecn
""".split())


def sin_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")


tematicas = proyectos[~proyectos["titulo"].str.upper().str.match(TRAMITE)]
od = tematicas["titulo"].str.upper().str.extract(r"^O\.?\s*D\.?\s*(\d+)")[0]
tematicas = tematicas.assign(proyecto=tematicas["anio"].astype(str) + "-" + od.fillna(tematicas["titulo"]))
por_proyecto = tematicas.groupby("proyecto").agg(titulos=("titulo", " ".join), idp_si=("idp_si", "mean"))
por_proyecto["lado"] = pd.cut(por_proyecto["idp_si"], [-np.inf, -UMBRAL_IDP, UMBRAL_IDP, np.inf],
                              labels=["izquierda", "centro", "derecha"])

formas = collections.defaultdict(collections.Counter)
frecuencias = {"izquierda": collections.Counter(), "derecha": collections.Counter()}
for titulos, lado in zip(por_proyecto["titulos"], por_proyecto["lado"]):
    if lado == "centro":
        continue
    palabras = set()
    for palabra in re.findall(r"[^\W\d_]{3,}", titulos.lower()):
        clave = sin_tildes(palabra)
        if clave not in NO_TEMA and not re.fullmatch(r"[ivxlc]+", clave):
            formas[clave][palabra] += 1
            palabras.add(clave)
    frecuencias[lado].update(palabras)

por_proyecto["lado"].value_counts()

In [ ]:
fig5, ejes = plt.subplots(1, 2, figsize=(14, 6), facecolor=PAGE)
for eje, (lado, color) in zip(ejes, [("izquierda", IZQUIERDA), ("derecha", DERECHA)]):
    pesos = {formas[clave].most_common(1)[0][0]: n for clave, n in frecuencias[lado].items()}
    nube = WordCloud(width=900, height=700, background_color=SURFACE, color_func=lambda *a, **k: color,
                     prefer_horizontal=1.0, random_state=42).generate_from_frequencies(pesos)
    eje.imshow(nube, interpolation="bilinear")
    eje.set_axis_off()
    n = (por_proyecto["lado"] == lado).sum()
    eje.set_title(f"Apoyados por la {lado} ({n} proyectos)", loc="left", fontsize=14, color=TEXT)
fig5.suptitle(f"Palabras en los títulos de los proyectos según quién los votó (|idp medio de los Sí| ≥ {UMBRAL_IDP})",
              x=0.01, ha="left", fontsize=16, color=TEXT)
plt.tight_layout()
plt.show()